# β Sensitivity Sweep — Posterior Variance Collapse

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/06_beta_sensitivity.ipynb)

This notebook reproduces **Example 1, Figure 1** from Alberts & Bilionis (2023).

We study the 1D steady-state heat equation

$$-D\phi''(x) = q(x), \quad x \in [0, 1]$$

with non-zero Dirichlet boundary conditions $\phi(0) = \phi_L$, $\phi(1) = \phi_R$ and
source $q(x) = e^{-x}$.  The field $\phi$ is represented in a Fourier basis that
automatically satisfies the boundary conditions.

**Goal:** sweep $\beta \in \{1, 10, 100, 1000\}$ and observe that the PIFT posterior
variance collapses toward the deterministic (FD) solution as $\beta \to \infty$.
Specifically, theory predicts $\text{Var}[\phi(x)] \propto 1/\beta$, so the product
$\text{Var} \cdot \beta$ should plateau toward a constant.

**Estimated runtime:** ~3 minutes on CPU (4 independent SGLD runs of 40k steps each).

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git

import time
import jax
jax.config.update('jax_enable_x64', True)
import numpy as np
import matplotlib.pyplot as plt

from pipelines.phase_b_beta_sweep import run_phase_b_beta_sweep

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    'beta_values':          [1.0, 10.0, 100.0, 1000.0],
    'D':                    0.25,       # diffusion coefficient              [1e-3, 10]
    'bc_left':              1.0,        # phi(0)                             [-10, 10]
    'bc_right':             0.1,        # phi(1)                             [-10, 10]
    'K':                    20,         # Fourier modes                      [4, 64]
    'n_quad':               96,         # quadrature points                  [16, 512]
    'n_grid':               300,        # plotting grid                      [50, 2000]
    'n_steps':              40000,      # SGLD steps per beta                [1000, 100000]
    'burn_in':              4000,       # warm-up                            [100, n_steps/2]
    'thin':                 8,          #                                    [1, 100]
    'step_size0':           1e-3,       # alpha_0 -- small for stability at beta=1000 [1e-5, 1e-2]
    'decay':                0.55,       #                                    [0.5, 1.0]
    'max_condition_number': 50.0,
}

## Run β Sweep

In [ ]:
t_start = time.perf_counter()

beta_result = run_phase_b_beta_sweep(
    cfg=CONFIG,
    device_preference=jax.default_backend(),
    save_outputs=False,
)

elapsed = time.perf_counter() - t_start
print(f'Status: {beta_result["status"]}')
print(f'Runtime: {elapsed:.1f} s  ({elapsed/60:.1f} min)')
print(f'Betas run: {[r["beta"] for r in beta_result["beta_results"]]}')
print(f'Samples per beta: {[r["n_samples"] for r in beta_result["beta_results"]]}')

## Results

In [ ]:
x_grid   = beta_result['x_grid']
phi_truth = beta_result['phi_truth']
results  = beta_result['beta_results']
n_betas  = len(results)

fig, axes = plt.subplots(1, n_betas, figsize=(4.5 * n_betas, 4.5), sharey=True)
if n_betas == 1:
    axes = [axes]

for ax, res in zip(axes, results):
    beta_val  = res['beta']
    phi_mean  = res['phi_mean']
    phi_std   = res['phi_std']
    phi_lo    = phi_mean - 1.645 * phi_std
    phi_hi    = phi_mean + 1.645 * phi_std

    ax.fill_between(x_grid, phi_lo, phi_hi, alpha=0.35, color='steelblue',
                    label='90% CI')
    ax.plot(x_grid, phi_mean, 'b-', lw=2, label='Posterior mean')
    ax.plot(x_grid, phi_truth, 'k--', lw=1.8, label='FD truth')

    var_mid = res['variance_at_midpoint']
    stop_flag = '  (early stop)' if res.get('stopped_early') else ''
    ax.set_title(f'$\\beta = {beta_val:.0f}$\nvar(0.5)={var_mid:.3e}{stop_flag}', fontsize=11)
    ax.set_xlabel('x')
    if ax is axes[0]:
        ax.set_ylabel('$\\phi(x)$')
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle(
    '$\\beta$ Sensitivity Sweep: Posterior Collapses as $\\beta \\to \\infty$',
    fontsize=13, y=1.02
)
fig.tight_layout()
show_fig(fig)

In [ ]:
print('Mean posterior variance vs \u03b2 (should fall ~1/\u03b2):')
vs = beta_result['variance_scaling']
if isinstance(vs, dict):
    for b, v, vt in zip(vs.get('betas', []), vs.get('variances', []), vs.get('var_times_beta', [])):
        print(f'  \u03b2 = {float(b):8.2f}    mean var = {float(v):.3e}    var\u00b7\u03b2 = {float(vt):.3f}')
else:
    betas = [r['beta'] for r in beta_result['beta_results']]
    for beta_val, var in zip(betas, vs):
        try:
            print(f'  \u03b2 = {beta_val:8.2f}    mean var = {float(var):.3e}')
        except (TypeError, ValueError):
            print(f'  \u03b2 = {beta_val:8.2f}    mean var = {var}')

## Interpretation

The table above should show the product $\text{Var}(\phi(0.5)) \cdot \beta$ plateauing
toward a roughly constant value as $\beta$ grows from 1 to 1000.  This is the **1/β
variance collapse** predicted by the PIFT theory (Alberts & Bilionis 2023, Proposition 2):
in the limit $\beta \to \infty$ the posterior concentrates around the deterministic
solution of the PDE, with variance shrinking as $1/\beta$.

The multi-panel plot shows the same effect visually: the 90% credible band (shaded blue)
tightens around the FD truth (dashed black) as $\beta$ increases.  At $\beta = 1$ the
posterior is nearly flat (very little physics information); at $\beta = 1000$ it has
essentially collapsed to the deterministic solution.